# 1-Instalacion e importacion de librerias

In [3]:
!pip install transformers

In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import os

# 2-Cargar el modelo

In [5]:
# Cargar el modelo y el tokenizador preentrenados
tokenizer = AutoTokenizer.from_pretrained("Clinical-AI-Apollo/Medical-NER")
model = AutoModelForTokenClassification.from_pretrained("Clinical-AI-Apollo/Medical-NER")

# Crear un pipeline para el reconocimiento de entidades nombradas
nlp = pipeline("ner", model=model, tokenizer=tokenizer)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/736M [00:00<?, ?B/s]

Device set to use cpu


## Funcion enriquecedora de texto

In [6]:
def enrich_text_with_ner(text):
    # Obtener los resultados del reconocimiento de entidades nombradas
    ner_results = nlp(text)

    # Ordenar las entidades por sus posiciones de inicio
    ner_results = sorted(ner_results, key=lambda x: x['start'])

    # Inicializar una lista para almacenar las entidades combinadas
    combined_entities = []

    # Combinar entidades contiguas del mismo tipo
    for entity in ner_results:
        entity_type = entity['entity'].split('-')[-1]  # Obtener el tipo de entidad sin el prefijo B- o I-
        if combined_entities and entity_type == combined_entities[-1]['entity']:
            combined_entities[-1]['end'] = entity['end']
            combined_entities[-1]['word'] += text[entity['start']:entity['end']]
        else:
            combined_entities.append({
                'entity': entity_type,
                'start': entity['start'],
                'end': entity['end'],
                'word': text[entity['start']:entity['end']]
            })

    # Inicializar una cadena vacía para almacenar el texto enriquecido
    enriched_text = ""

    # Inicializar una variable para rastrear la última posición en el texto
    last_position = 0

    # Iterar sobre las entidades combinadas y enriquecer el texto
    for entity in combined_entities:
        # Agregar el texto antes de la entidad
        enriched_text += text[last_position:entity['start']]

        # Agregar la entidad con etiquetas
        enriched_text += f"<{entity['entity']}>{entity['word']}</{entity['entity']}>"

        # Actualizar la última posición
        last_position = entity['end']

    # Agregar el texto restante después de la última entidad
    enriched_text += text[last_position:]

    return enriched_text

# 3-Aplicar el enriquecedor al corpus:
Subir el archivo a la ruta /content/

In [8]:
path = '/content/'
file = 'MED.ALL'
pathFile = os.path.join(path,file)
pathEnrichedFile = os.path.join(path,'MED_ENRICHED.ALL')

aplica el enriquecedor al corpus, linea a linea

In [17]:
#linea_inicio = 0
linea_inicio = 22621

# Abrir el archivo de entrada y leer todas las lineas
with open(pathFile, 'r', encoding='utf-8') as archivo_entrada:
    lineas = archivo_entrada.readlines()

# Leer el archivo de salida existente (si existe)
try:
    with open(pathEnrichedFile, 'r', encoding='utf-8') as archivo_salida:
        lineas_salida = archivo_salida.readlines()
except FileNotFoundError:
    lineas_salida = []  # Si no existe, inicializar como lista vacia

# Abrir el archivo de salida en modo de escritura (sobrescribir)
with open(pathEnrichedFile, 'w', encoding='utf-8') as archivo_salida:
    # Escribir las líneas anteriores que no se van a sobrescribir
    for i in range(min(linea_inicio, len(lineas_salida))):
        archivo_salida.write(lineas_salida[i])

    # Procesar las lineas desde la línea de inicio
    for i in range(linea_inicio, len(lineas)):
        linea = lineas[i].strip()  # Eliminar espacios en blanco al inicio y al final

        # Verificar si la linea no está vacia
        if linea:
            # Aplicar la función de enriquecimiento
            if not (linea.startswith('.I') or linea.startswith('.W')):
                linea_enriquecida = enrich_text_with_ner(linea)
            else:
                linea_enriquecida = linea

            # Escribir la linea enriquecida en el archivo de salida
            archivo_salida.write(linea_enriquecida + '\n')


In [18]:
# Descargar el archivo
from google.colab import files
files.download(pathEnrichedFile)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>